# Epigenetic marks from the shared channels: accuracy and reliable relations (Jackson → Danenberg, Keren)

**Question.** Can models infer Jackson's epigenetic marks from the 14 channels both cohorts share
(H3, DNA1, DNA2, ERα, HER2, Ki-67, CD3, CD20, CD45, CD68, CK5, CK8/18, panCK, SMA)?
- **H3K27me3:** the repressive chromatin mark (La139).
- **pHH3:** histone H3 phosphorylation (Eu153), which marks mitosis.
- Danenberg measured neither mark. There, an imputed mark can only be judged by whether it keeps the relations with the
  shared markers (and clinical labels) that are reliable in Jackson.

**Models** (all trained on the Jackson train patients of replicate 0, the split of `immuvis-jackson-train`):
- **LightGBM, nuclear only:** H3, DNA1, DNA2 and cell area. The floor: what nuclear content alone gives.
- **LightGBM:** the 14 shared channels and cell area (cohort-wide rank-normalised, as recommended in `cell-table-followup`).
- **ImmuVis** (generative masked autoencoder; `epi-immuvis-train` → `epi-immuvis-infer`): both marks from the 14 shared channels.
- **EVA, zero-shot** (`epi-immuvis-infer`): within images only, because it scales every channel per image.
- **Leave-anchor-out (LOA):** LightGBM refitted without marker M, and ImmuVis given the 14 channels minus M.

**Pre-registered rules** (written 2026-10-21, before any result of this kernel or of `epi-immuvis-*`):

*Relations* (tumour images; values arcsinh(x / 1), then normal scores within the dataset; nuclear content = H3, DNA1, DNA2 and log area):
- **C, cell level:** partial correlation between the mark and marker M within images, given nuclear content.
  All cells, and within epithelial, immune and stromal cells (2 marks × 11 markers × 4 = 88).
- **K, cell class:** within-image difference of the nuclear-adjusted mark, immune − epithelial and stromal − epithelial,
  in within-image SD units (4).
- **P, patient level:** Spearman correlation across patients between the epithelial-cell means of the nuclear-adjusted mark
  and of M (patients with ≥ 50 epithelial cells; 22).
- **L, clinical:** AUROC − 0.5 of the patient's nuclear-adjusted mark (epithelial cells) for ER−, PR−, grade 3 and
  triple-negative (8). These labels are not model inputs (ER and HER2 status are related to inputs).

*Reliable in Jackson* (all must hold):
1. the 95% patient-bootstrap CI over all Jackson patients excludes 0;
2. |effect| ≥ 0.10 (C), 0.20 (K, P) or 0.06 (L);
3. the same sign in Basel and in Zurich, separately;
4. C and K only: with 2-px eroded masks the effect keeps its sign and at least half its size
   (compared with the same image-derived values under full masks). This guards against spillover from neighbours.

*Reproduced by an imputed mark:* the same sign as the reference, a 95% CI excluding 0, and effect / reference within 0.5–2.
The reference is the measured mark on the same cells (Jackson test patients) or the Jackson effect (Danenberg).

*Circularity.* For C and P relations, the anchor M is an input of the full models, so the relation can be copied from the
input. The verdict therefore uses the LOA predictions (models that never saw M) for C and P, and the full predictions
for K and L. Results with the full predictions are reported too.

*Verdict per model and mark:*
- **A, accuracy:** partial correlation with the measured mark beyond nuclear content (within images, Jackson test
  patients) ≥ 0.30, with CI > 0.
- **B:** ≥ 2/3 of the reliable relations reproduced in the Jackson test patients.
- **C:** ≥ 2/3 reproduced in Danenberg.
- **"infers"** = A + B + C; **"only within Jackson"** = A + B; **"no"** otherwise.
  EVA is judged on C and K relations only, and was not run on Danenberg.

*Added after the local smoke test (reported only, not part of the rules):* for P relations, the Spearman correlation
after removing each patient's overall signal level (mean normal score of the 14 shared channels), because in the smoke
sample the mark correlated positively with almost every marker across patients.

---
**Amendment 1 (2026-10-22, after version 2; it changes how the Danenberg check is read, not the original verdict, which
is still reported).** In version 2, the Danenberg relations that failed were concentrated on anchors that do not transfer
between the cohorts: every SMA relation, most CD68 relations and the CD45 patient relation. The anchor itself is measured
differently there, so its relation with an imputed mark cannot be expected to hold (`train-test-grid`: a Jackson-trained
SMA model scores R4 = −0.17 in Danenberg). This rule was written after seeing the Danenberg results, so it is labelled
post hoc.
- **Transferable anchor:** R4 in the Danenberg test patients when the marker is predicted by a Jackson-trained model,
  divided by R4 when trained on Danenberg (`train-test-grid`, Table model, cell-mean truth, mean of 3 replicates)
  ≥ 0.60. This excludes SMA (−0.21), CD68 (0.43) and CD20 (0.51); the next lowest is CK5 (0.73).
- **C′:** ≥ 2/3 of the reliable relations reproduced in Danenberg, counting only C and P relations with a transferable
  anchor, plus every K and L relation (these have no anchor). The verdict "infers" with C′ in place of C is reported as
  `verdict_amended`.

**Keren 2018 (pre-registered 2026-10-22, before any Keren result).** Keren (TNBC, MIBI-TOF, 40 patients, one image each;
`keren-cells-prep`) measured H3K27me3 with the same clone as Jackson (C36B11). It is the only cohort outside Jackson where
an imputed H3K27me3 can be checked against the measured mark. It has no pHH3.
- **Inputs:** the 7 markers Keren shares with Jackson and Danenberg (Ki-67, CD3, CD20, CD45, CD68, SMA, panCK), dsDNA in
  place of DNA1 (both DNA intercalator stains), and log area. Keren has no H3, DNA2, ERα, HER2, CK5 or CK8/18.
  Nuclear content in every Keren analysis (and in the Jackson references it is compared with) = DNA1/dsDNA and log area.
- **Models:** LightGBM (nuclear only; the 9 Keren inputs; leave-anchor-out), trained on the same Jackson train patients
  with features rank-normalised within each cohort. The image models need the 14 channels and are not run on Keren.
- **Relations:** C (7 anchors × 4 strata), K (2) and P (7) for H3K27me3, with the reduced nuclear covariates. Reliable in
  Jackson by the same four criteria as above. No L: every Keren patient is triple-negative.
- **Transferable to Keren:** for each anchor M, within-image r in Keren of M predicted by a Jackson-trained LightGBM
  (the other Keren inputs) ÷ the same r with a Keren-trained LightGBM (5-fold, by patient) ≥ 0.60.

  This is the same rule as Amendment 1, but computed here because there is no grid for Keren.
- **Reported:**
  - **Biological replication:** does the *measured* Keren mark show each reliable Jackson relation? Same sign, CI excluding 0, effect / Jackson effect within 0.5–2.
  - **Truth-referenced fidelity:** for each imputed mark, the relation is computed on the same Keren cells and judged against the measured Keren effect (the Jackson-test rule).
  - **Jackson-referenced fidelity:** the same relation judged against the Jackson effect (the Danenberg rule).
  - **Agreement between the two:** tests whether the Danenberg rule gives the right answer where the truth is known.
  - For reference only: the Jackson effect in Jackson's triple-negative patients, and a 2 µm erosion check of the measured Keren relations.
- **Keren verdict per model:**
  - **A_K:** partial correlation with the measured mark beyond nuclear content (within images, Keren) ≥ 0.30, with CI > 0.
  - **D_K:** ≥ 2/3 of the reliable relations with a transferable anchor (and K) reproduced against the measured Keren mark (primary predictions: leave-anchor-out for C and P).
  - **Outcome:** "infers in Keren" = A_K + D_K. "accurate, relations not kept" = A_K only. "no" otherwise.


In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback, gc
import numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy import ndimage, stats
from scipy.special import ndtri
from sklearn.model_selection import GroupKFold
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"          # small, fast settings for a local test (synthetic image predictions)
SEED = 20261021
SPLIT_SEED = 20261013       # Jackson split of immuvis-jackson-train (replicate 0 of train-test-grid)
COFACTOR = 1.0
MIN_CELLS = 200
MIN_PAT_CELLS = 50
MIN_IMG_CLASS = 20
ERODE_PX = 2
N_BOOT = 100 if SMOKE else 1000
N_BOOT_AUC = 50 if SMOKE else 200
N_TRAIN = 20_000 if SMOKE else 300_000
N_FOLD = 5
N_JOBS = os.cpu_count()
LGB = dict(n_estimators=30 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=N_JOBS, verbose=-1)
NUC = ["histone_h3", "dna1", "dna2"]
SHARED = {   # Jackson name -> Danenberg name (HER2 is clone 3B5 in both)
    "estrogen_receptor_alpha": "estrogen_receptor_alpha", "c_erb_b_2_her2": "c_erb_b_2_her2_3b5",
    "ki_67": "ki_67", "cd3": "cd3", "cd20": "cd20", "cd45": "cd45", "cd68": "cd68",
    "cytokeratin_5": "cytokeratin_5", "cytokeratin_8_18": "cytokeratin_8_18",
    "pan_cytokeratin": "pan_cytokeratin", "sma": "sma"}
MARKERS = list(SHARED)
EPI = {"h3k27me3": "histone_h3_trimethylate", "phh3": "histone_h3_phospho"}
EPI_LABEL = {"h3k27me3": "H3K27me3", "phh3": "pHH3"}
LABEL = {"estrogen_receptor_alpha": "ERα", "c_erb_b_2_her2": "HER2", "ki_67": "Ki-67", "cd3": "CD3", "cd20": "CD20",
         "cd45": "CD45", "cd68": "CD68", "cytokeratin_5": "CK5", "cytokeratin_8_18": "CK8/18", "pan_cytokeratin": "panCK",
         "sma": "SMA"}
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
CLASSES = ("epithelial", "immune", "stromal")
CLIN = {"er_neg": "ER−", "pr_neg": "PR−", "grade3": "grade 3", "tnbc": "triple-negative"}
THR = {"C": 0.10, "K": 0.20, "P": 0.20, "L": 0.06}
RATIO = (0.5, 2.0)
ACC_MIN = 0.30
REPRO_FRAC = 2 / 3
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
MODEL_NAME = {"nuclear": "LightGBM, nuclear only", "lgbm": "LightGBM", "immuvis": "ImmuVis", "eva": "EVA"}
T0 = time.time()
FAILED = []
OUT_JSON = {"smoke": SMOKE}

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

def read_parquet(path, columns=None):
    try:
        return pd.read_parquet(path, columns=columns)
    except Exception:            # local pyarrow cannot read some Kaggle-written files; duckdb can
        import duckdb
        cols = "*" if columns is None else ", ".join(f'"{c}"' for c in columns)
        return duckdb.query(f"select {cols} from read_parquet('{path}')").df()

def ci(v):
    v = np.asarray(v, np.float64)
    v = v[np.isfinite(v)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) else (np.nan, np.nan)

def nscore(v):
    v = np.asarray(v, np.float64)
    out = np.full(len(v), np.nan)
    ok = np.isfinite(v)
    out[ok] = ndtri((stats.rankdata(v[ok]) - 0.5) / ok.sum())
    return out

def demean(X, img):
    X = np.asarray(X, np.float64)
    k = int(img.max()) + 1
    cnt = np.maximum(np.bincount(img, minlength=k), 1).astype(np.float64)
    if X.ndim == 1:
        return X - (np.bincount(img, X, k) / cnt)[img]
    return X - np.column_stack([np.bincount(img, X[:, j], k) / cnt for j in range(X.shape[1])])[img]

def rank_norm(X):
    return np.column_stack([(stats.rankdata(X[:, j]) - 0.5) / len(X) for j in range(X.shape[1])]).astype(np.float32)

## Cells and clinical labels

In [ ]:
P = {}
for p in g("cells.parquet"):
    names = pq.read_schema(p).names
    if "object_number" not in names or "cell_phenotype" not in names:
        continue
    key = "danenberg" if "metabric_id" in names else "keren" if "published__h3k27me3" in names else "jackson"
    P[key] = dict(cells=p, clinical=near("clinical.parquet", p))
print(json.dumps(P, indent=1))
assert {"danenberg", "jackson"} <= set(P), P

def load(key):
    if key == "jackson":
        c = read_parquet(P[key]["cells"], ["image_name", "object_number", "patient_id", "cohort", "is_tumour", "broad_class",
                                           "cell_phenotype", "area"] + NUC + MARKERS + list(EPI.values()))
        c = c[c.is_tumour == 1].copy()
        c["patient"], c["sub"], c["broad"] = c.patient_id.astype(str), c.cohort.astype(str), c.broad_class.astype(str)
    else:
        c = read_parquet(P[key]["cells"], ["image_name", "object_number", "metabric_id", "is_tumour", "is_hot_aggregate",
                                           "is_epithelial", "cell_phenotype", "area"] + NUC + list(SHARED.values()))
        c = c[(c.is_tumour == 1) & (c.is_hot_aggregate == 0)].copy()
        c["patient"], c["sub"] = c.metabric_id.astype(str), "metabric"
        epi_frac = c.groupby("cell_phenotype").is_epithelial.mean()
        bmap = {p: "epithelial" if epi_frac[p] > 0.5 else
                "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
        c["broad"] = c.cell_phenotype.map(bmap).astype(str)
        c = c.rename(columns={v: k for k, v in SHARED.items() if v != k})
    n_per = c.groupby("image_name").size()
    c = c[c.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    if SMOKE:
        keep = np.random.default_rng(1).choice(c.image_name.unique(), 80, replace=False)
        c = c[c.image_name.isin(keep)]
    c = c.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    chans = NUC + MARKERS + (list(EPI.values()) if key == "jackson" else [])
    D = dict(key=key, cells=c[["image_name", "object_number", "patient", "sub", "broad"]].copy(),
             A={ch: np.arcsinh(c[ch].values.astype(np.float64) / COFACTOR).astype(np.float32) for ch in chans},
             log_area=np.log(np.maximum(c["area"].values.astype(np.float64), 1)).astype(np.float32),
             img=pd.factorize(c.image_name, sort=True)[0].astype(np.int64), broad=c.broad.values.astype(str),
             sub=c["sub"].values.astype(str))
    D["pats"], D["pcode"] = np.unique(D["cells"].patient.values, return_inverse=True)
    D["n"] = len(c)
    print(f"{key}: {len(c):,} cells, {c.image_name.nunique()} images, {len(D['pats'])} patients, "
          f"classes {pd.Series(D['broad']).value_counts().to_dict()}")
    return D

def clinical(D):
    c = read_parquet(P[D["key"]]["clinical"]).astype(str).replace({"nan": np.nan, "[]": np.nan, "None": np.nan})
    if D["key"] == "jackson":
        her2 = c.HER2Status.map({"positive": 1.0, "negative": 0.0}).fillna(c.HER2.map({"+": 1.0, "-": 0.0}))
        lab = pd.DataFrame({"pid": c.patient_id.values, "er_neg": c.ERStatus.map({"negative": 1.0, "positive": 0.0}).values,
                            "pr_neg": c.PRStatus.map({"negative": 1.0, "positive": 0.0}).values,
                            "grade3": c.grade.map({"3": 1.0, "2": 0.0, "1": 0.0}).values, "her2_pos": her2.values})
    else:
        lab = pd.DataFrame({"pid": c.metabric_id.values, "er_neg": c.ER_STATUS.map({"Negative": 1.0, "Positive": 0.0}).values,
                            "pr_neg": c.PR_STATUS.map({"Negative": 1.0, "Positive": 0.0}).values,
                            "grade3": c.GRADE.map({"3": 1.0, "2": 0.0, "1": 0.0}).values,
                            "her2_pos": c.HER2_STATUS.map({"Positive": 1.0, "Negative": 0.0}).values})
    lab = lab.groupby("pid").first()
    known = lab[["er_neg", "pr_neg", "her2_pos"]].notna().all(1)
    lab["tnbc"] = np.where(known, ((lab.er_neg == 1) & (lab.pr_neg == 1) & (lab.her2_pos == 0)).astype(float), np.nan)
    lab = lab.reindex(D["pats"])
    print(D["key"], "clinical labels (patients with a label, positives):",
          {k: (int(lab[k].notna().sum()), int(np.nansum(lab[k]))) for k in CLIN})
    return {k: lab[k].values.astype(np.float64) for k in CLIN}

DJ, DD = load("jackson"), load("danenberg")
DJ["clin"], DD["clin"] = clinical(DJ), clinical(DD)
_u = np.random.default_rng(SPLIT_SEED + 11 + 0).random(len(DJ["pats"]))
DJ["split"] = np.where(_u < 0.7, "train", np.where(_u < 0.8, "val", "test"))[DJ["pcode"]]
print("Jackson split (cells):", pd.Series(DJ["split"]).value_counts().to_dict(),
      "| test patients:", len(np.unique(DJ["pcode"][DJ["split"] == "test"])))
OUT_JSON["cells"] = {"jackson": DJ["n"], "danenberg": DD["n"]}
tick("tables loaded")

## Jackson per-cell means from the images under full and 2-px eroded masks (spillover check)

In [ ]:
IMG = {}
try:
    try:
        import imagecodecs  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs==2024.9.22"],
                       check=True)
    import tifffile
    tifs = g("*.tiff")
    J_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
             if not os.path.basename(p).startswith("MB_") and "/masks/" not in p.replace("\\", "/") and "jackson" in p.lower()}
    J_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
              if not os.path.basename(p).startswith("MB_") and "/masks/" in p.replace("\\", "/")}
    jix = {m: i for i, m in enumerate(J_CHAN)}
    ER_CH = NUC + MARKERS + list(EPI.values())
    s_ = DJ["cells"].image_name.values
    st = np.r_[0, np.flatnonzero(s_[1:] != s_[:-1]) + 1, len(s_)]
    SPAN = {s_[a]: (a, b) for a, b in zip(st[:-1], st[1:])}
    todo = [s for s in SPAN if s in J_IMG and s in J_MASK]
    print(f"{len(todo)} of {len(SPAN)} Jackson images have a TIFF and a mask")
    assert todo, "no Jackson images available"

    def erode_labels(lab, r):
        mx = ndimage.maximum_filter(lab, size=2 * r + 1, mode="constant", cval=0)
        mn = ndimage.minimum_filter(lab, size=2 * r + 1, mode="constant", cval=0)
        return np.where((mx == lab) & (mn == lab), lab, 0)

    def img_means(s):
        a, b = SPAN[s]
        obj = DJ["cells"].object_number.values[a:b].astype(np.int64)
        im = tifffile.imread(J_IMG[s])
        if im.ndim == 3 and im.shape[0] == len(J_CHAN) and im.shape[-1] != len(J_CHAN):
            im = np.moveaxis(im, 0, -1)
        lab = tifffile.imread(J_MASK[s]).astype(np.int64)
        assert lab.shape == im.shape[:2], (s, lab.shape, im.shape)
        res = {}
        for tag, L in (("full", lab), ("eroded", erode_labels(lab, ERODE_PX))):
            lut = np.full(max(int(L.max()), int(obj.max())) + 1, -1, np.int64)
            lut[obj] = np.arange(len(obj))
            loc = lut[L].ravel()
            v = loc >= 0
            cnt = np.bincount(loc[v], minlength=len(obj))
            M = np.full((len(obj), len(ER_CH)), np.nan, np.float32)
            for j, ch in enumerate(ER_CH):
                sm = np.bincount(loc[v], im[..., jix[ch]].ravel()[v].astype(np.float64), len(obj))
                M[:, j] = np.where(cnt > 0, sm / np.maximum(cnt, 1), np.nan)
            res[tag] = (M, cnt)
        return s, res

    for tag in ("full", "eroded"):
        IMG[tag] = {ch: np.full(DJ["n"], np.nan, np.float32) for ch in ER_CH}
        IMG[tag]["n_px"] = np.zeros(DJ["n"], np.float32)
    for s, res in Parallel(n_jobs=min(4, N_JOBS))(delayed(img_means)(s) for s in todo):
        a, b = SPAN[s]
        for tag, (M, cnt) in res.items():
            for j, ch in enumerate(ER_CH):
                IMG[tag][ch][a:b] = np.arcsinh(M[:, j] / COFACTOR)
            IMG[tag]["n_px"][a:b] = cnt
    for tag in IMG:
        IMG[tag]["log_area"] = np.log(np.maximum(IMG["full"]["n_px"], 1)).astype(np.float32)
    ok = np.isfinite(IMG["full"][EPI["h3k27me3"]])
    OUT_JSON["erosion"] = dict(images=len(todo), cells_with_full_mask_values=int(ok.sum()),
                               cells_kept_after_erosion=int(np.isfinite(IMG["eroded"][EPI["h3k27me3"]]).sum()),
                               median_pixels_full=float(np.median(IMG["full"]["n_px"][ok])),
                               median_pixels_eroded=float(np.median(IMG["eroded"]["n_px"][ok])),
                               table_vs_image_r={ch: float(np.corrcoef(DJ["A"][ch][ok], IMG["full"][ch][ok])[0, 1])
                                                 for ch in ["dna1", EPI["h3k27me3"], EPI["phh3"], "cd45"]})
    print(json.dumps(OUT_JSON["erosion"], indent=1))
    tick("image means under full and eroded masks")
except Exception:
    print("!!! erosion block failed:\n" + traceback.format_exc(), flush=True)
    FAILED.append("erosion")
    IMG = {}
    if SMOKE:                       # local test without TIFFs: synthetic stand-in, only to exercise the code path
        print("SMOKE: synthetic eroded values (table + noise); not a result")
        rs = np.random.default_rng(2)
        IMG = {t: {**{ch: DJ["A"][ch] + (rs.normal(0, 0.2, DJ["n"]) if t == "eroded" else 0) for ch in
                      NUC + MARKERS + list(EPI.values())}, "n_px": np.full(DJ["n"], 50.0), "log_area": DJ["log_area"]}
               for t in ("full", "eroded")}

## Relation statistics

In [ ]:
class View:
    """A set of rows of one cohort. Every array used with a view must be finite on its rows.
    src: full-length arrays for the nuclear covariates and the shared markers (measured values)."""
    def __init__(self, D, rows, name, src=None, nuc=None):
        self.D, self.rows, self.name = D, np.asarray(rows), name
        self.src = src if src is not None else {**D["A"], "log_area": D["log_area"]}
        self.img = pd.factorize(D["img"][self.rows])[0].astype(np.int64)
        self.pcode = D["pcode"][self.rows]
        self.n_pat = len(D["pats"])
        self.broad = D["broad"][self.rows]
        self.clin = D["clin"]
        self.N = np.column_stack([nscore(self.src[c][self.rows]) for c in (nuc or NUC + ["log_area"])])
        assert np.isfinite(self.N).all(), f"{name}: nuclear covariates not finite"
        act = np.unique(self.pcode)
        rng = np.random.default_rng(SEED + 7)
        self.W = np.zeros((N_BOOT, self.n_pat))
        self.W[:, act] = rng.multinomial(len(act), np.full(len(act), 1 / len(act)), size=N_BOOT)
        self._q, self._ns = {}, {}

    def ns_src(self, ch):
        if ch not in self._ns:
            self._ns[ch] = nscore(self.src[ch][self.rows])
        return self._ns[ch]

    def ns(self, v):
        x = nscore(np.asarray(v)[self.rows])
        assert np.isfinite(x).all(), f"{self.name}: values not finite on the view rows"
        return x

    def _resid(self, stratum):
        if stratum not in self._q:
            idx = np.flatnonzero(np.ones(len(self.rows), bool) if stratum == "all" else self.broad == stratum)
            im = pd.factorize(self.img[idx])[0].astype(np.int64)
            Q, _ = np.linalg.qr(demean(self.N[idx], im))
            self._q[stratum] = (idx, im, Q)
        return self._q[stratum]

    def adjusted(self, x, stratum):
        """Within-image residual of x after the nuclear covariates (rows of the stratum)."""
        idx, im, Q = self._resid(stratum)
        r = demean(x[idx], im)
        return r - Q @ (Q.T @ r), idx, im

    def cell_rel(self, e, m, stratum):
        e_, idx, _ = self.adjusted(e, stratum)
        m_, _, _ = self.adjusted(m, stratum)
        pc = self.pcode[idx]
        sxy, sxx, syy = (np.bincount(pc, w, self.n_pat) for w in (e_ * m_, e_ * e_, m_ * m_))
        return float(sxy.sum() / np.sqrt(sxx.sum() * syy.sum())), (self.W @ sxy) / np.sqrt((self.W @ sxx) * (self.W @ syy))

    def r_within(self, a, b):
        """Plain within-image Pearson r (no covariates)."""
        a_, b_ = demean(np.asarray(a, np.float64)[self.rows], self.img), demean(np.asarray(b, np.float64)[self.rows], self.img)
        sxy, sxx, syy = (np.bincount(self.pcode, w, self.n_pat) for w in (a_ * b_, a_ * a_, b_ * b_))
        return float(sxy.sum() / np.sqrt(sxx.sum() * syy.sum())), (self.W @ sxy) / np.sqrt((self.W @ sxx) * (self.W @ syy))

    def class_rel(self, e, a, b):
        e_, idx, im = self.adjusted(e, "all")
        sd = np.sqrt(np.mean(e_ ** 2))
        br = self.broad[idx]
        k = int(im.max()) + 1
        ia, ib = (br == a).astype(np.float64), (br == b).astype(np.float64)
        na, nb = np.bincount(im, ia, k), np.bincount(im, ib, k)
        ok = (na >= MIN_IMG_CLASS) & (nb >= MIN_IMG_CLASS)
        if not ok.any():
            return np.nan, np.full(N_BOOT, np.nan)
        d = (np.bincount(im, e_ * ia, k)[ok] / na[ok] - np.bincount(im, e_ * ib, k)[ok] / nb[ok]) / sd
        ip = np.zeros(k, np.int64)
        ip[im] = self.pcode[idx]
        sd_, sc = np.bincount(ip[ok], d, self.n_pat), np.bincount(ip[ok], minlength=self.n_pat).astype(np.float64)
        return float(sd_.sum() / sc.sum()), (self.W @ sd_) / (self.W @ sc)

    def patient_mean(self, x):
        idx = np.flatnonzero(self.broad == "epithelial")
        pc = self.pcode[idx]
        cnt = np.bincount(pc, minlength=self.n_pat)
        return np.where(cnt >= MIN_PAT_CELLS, np.bincount(pc, x[idx], self.n_pat) / np.maximum(cnt, 1), np.nan)

    def brightness(self):
        """Patient mean (epithelial cells) of the average normal score over the 14 shared channels: overall signal level."""
        if not hasattr(self, "_bright"):
            self._bright = self.patient_mean(np.mean([self.ns_src(c) for c in NUC + MARKERS], axis=0))
        return self._bright

    def patient_adj(self, e):
        """Patient mean (epithelial cells) of the mark after a pooled linear adjustment for nuclear content."""
        idx = np.flatnonzero(self.broad == "epithelial")
        Z = np.column_stack([np.ones(len(idx)), self.N[idx]])
        r = np.full(len(e), np.nan)
        r[idx] = e[idx] - Z @ np.linalg.lstsq(Z, e[idx], rcond=None)[0]
        return self.patient_mean(np.nan_to_num(r))

def _boot_idx(n, salt):
    return np.random.default_rng(SEED + salt).integers(0, n, (N_BOOT, n))

def spearman_boot(x, y):
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 10:
        return np.nan, np.full(N_BOOT, np.nan)
    BI = _boot_idx(len(x), 3)
    rx, ry = stats.rankdata(x[BI], axis=1), stats.rankdata(y[BI], axis=1)
    rx -= rx.mean(1, keepdims=True)
    ry -= ry.mean(1, keepdims=True)
    return float(stats.spearmanr(x, y).correlation), (rx * ry).sum(1) / np.sqrt((rx ** 2).sum(1) * (ry ** 2).sum(1))

def partial_spearman_boot(x, y, z):
    """Spearman correlation of x and y after removing the rank of z from both (linear on ranks)."""
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x, y, z = x[ok], y[ok], z[ok]
    if len(x) < 10:
        return np.nan, np.full(N_BOOT, np.nan)
    def pr(rx, ry, rz):
        rx, ry, rz = (r - r.mean(-1, keepdims=True) for r in (rx, ry, rz))
        rx = rx - (rx * rz).sum(-1, keepdims=True) / (rz * rz).sum(-1, keepdims=True) * rz
        ry = ry - (ry * rz).sum(-1, keepdims=True) / (rz * rz).sum(-1, keepdims=True) * rz
        return (rx * ry).sum(-1) / np.sqrt((rx ** 2).sum(-1) * (ry ** 2).sum(-1))
    BI = _boot_idx(len(x), 4)
    point = pr(*(stats.rankdata(v) for v in (x, y, z)))
    return float(point), pr(*(stats.rankdata(v[BI], axis=1) for v in (x, y, z)))

def _auc(s, l):
    r = stats.rankdata(s, axis=-1)
    npos = l.sum(-1)
    nneg = l.shape[-1] - npos
    with np.errstate(invalid="ignore", divide="ignore"):
        return ((r * l).sum(-1) - npos * (npos + 1) / 2) / (npos * nneg)

def auc_boot(score, label):
    ok = np.isfinite(score) & np.isfinite(label)
    s, l = score[ok], label[ok]
    if len(s) < 10 or l.sum() < 3 or (1 - l).sum() < 3:
        return np.nan, np.full(N_BOOT, np.nan)
    BI = _boot_idx(len(s), 5)
    return float(_auc(s, l) - 0.5), _auc(s[BI], l[BI]) - 0.5

def rel_value(V, rel, e_full):
    """Effect and bootstrap draws of relation rel for the mark values e_full (full-length array) on view V."""
    e = V.ns(e_full)
    f = rel["family"]
    if f == "C":
        return V.cell_rel(e, V.ns_src(rel["anchor"]), rel["stratum"])
    if f == "K":
        return V.class_rel(e, rel["a"], rel["b"])
    if f == "P":
        return spearman_boot(V.patient_adj(e), V.patient_mean(V.ns_src(rel["anchor"])))
    return auc_boot(V.patient_adj(e), V.clin[rel["label"]])

RELS = []
for mk in EPI:
    for M in MARKERS:
        for stratum in ("all",) + CLASSES:
            RELS.append(dict(id=f"C|{mk}|{M}|{stratum}", family="C", mark=mk, anchor=M, stratum=stratum))
    for a, b in (("immune", "epithelial"), ("stromal", "epithelial")):
        RELS.append(dict(id=f"K|{mk}|{a}-{b}", family="K", mark=mk, anchor=None, a=a, b=b))
    for M in MARKERS:
        RELS.append(dict(id=f"P|{mk}|{M}", family="P", mark=mk, anchor=M))
    for lab in CLIN:
        RELS.append(dict(id=f"L|{mk}|{lab}", family="L", mark=mk, anchor=None, label=lab))

def describe(rel):
    m = EPI_LABEL[rel["mark"]]
    if rel["family"] == "C":
        return f"{m} ~ {LABEL[rel['anchor']]}, cells ({rel['stratum']})"
    if rel["family"] == "K":
        return f"{m}: {rel['a']} − {rel['b']} cells"
    if rel["family"] == "P":
        return f"{m} ~ {LABEL[rel['anchor']]}, patients"
    return f"{m} vs {CLIN[rel['label']]}"
print(len(RELS), "candidate relations")

## Phase A: which relations are reliable in Jackson (measured marks)

In [ ]:
VJ = View(DJ, np.arange(DJ["n"]), "Jackson, all patients")
VB = View(DJ, np.flatnonzero(DJ["sub"] == "basel"), "Jackson, Basel")
VZ = View(DJ, np.flatnonzero(DJ["sub"] == "zurich"), "Jackson, Zurich")
VJF = VJE = None
if IMG:
    ok_e = np.isfinite(np.column_stack([IMG["eroded"][c] for c in IMG["eroded"] if c != "n_px"])).all(1) & \
           np.isfinite(np.column_stack([IMG["full"][c] for c in IMG["full"] if c != "n_px"])).all(1)
    VJF = View(DJ, np.flatnonzero(ok_e), "Jackson, image values, full masks", src=IMG["full"])
    VJE = View(DJ, np.flatnonzero(ok_e), "Jackson, image values, eroded masks", src=IMG["eroded"])
rows = []
for rel in RELS:
    ch = EPI[rel["mark"]]
    pt, bt = rel_value(VJ, rel, DJ["A"][ch])
    pb, _ = rel_value(VB, rel, DJ["A"][ch])
    pz, _ = rel_value(VZ, rel, DJ["A"][ch])
    row = dict(id=rel["id"], family=rel["family"], mark=rel["mark"], relation=describe(rel), effect=pt,
               lo=ci(bt)[0], hi=ci(bt)[1], basel=pb, zurich=pz, full_img=np.nan, eroded=np.nan)
    if rel["family"] in "CK" and VJE is not None:
        row["full_img"], _ = rel_value(VJF, rel, IMG["full"][ch])
        row["eroded"], _ = rel_value(VJE, rel, IMG["eroded"][ch])
    if rel["family"] == "P":      # added after the smoke test, reported only: patient-level overall signal level
        e = VJ.ns(DJ["A"][ch])
        pa, ba = partial_spearman_boot(VJ.patient_adj(e), VJ.patient_mean(VJ.ns_src(rel["anchor"])), VJ.brightness())
        row.update(brightness_adj=pa, brightness_adj_lo=ci(ba)[0], brightness_adj_hi=ci(ba)[1])
    rows.append(row)
REL = pd.DataFrame(rows)
sgn = np.sign(REL.effect)
REL["ci_excludes_0"] = (REL.lo > 0) | (REL.hi < 0)
REL["large_enough"] = REL.effect.abs() >= REL.family.map(THR)
REL["same_sign_basel_zurich"] = (np.sign(REL.basel) == sgn) & (np.sign(REL.zurich) == sgn)
REL["erosion_ok"] = np.where(REL.family.isin(["C", "K"]),
                             (np.sign(REL.eroded) == sgn) & (np.sign(REL.full_img) == sgn) &
                             (REL.eroded.abs() >= 0.5 * REL.full_img.abs()), True)
REL["reliable"] = REL.ci_excludes_0 & REL.large_enough & REL.same_sign_basel_zurich & REL.erosion_ok
REL.to_csv(f"{OUT}/relations_jackson.csv", index=False)
OUT_JSON["n_reliable"] = REL[REL.reliable].groupby(["mark", "family"]).size().unstack(fill_value=0).to_dict(orient="index")
print("reliable relations by mark and family:", OUT_JSON["n_reliable"])
print(REL[REL.reliable].sort_values(["mark", "family", "effect"])[["relation", "effect", "lo", "hi", "basel", "zurich",
                                                                  "full_img", "eroded"]].round(3).to_string(index=False))
rp = REL[REL.reliable & (REL.family == "P")]
OUT_JSON["reliable_P_surviving_brightness_adjustment"] = {
    mk: f"{int(((np.sign(d.brightness_adj) == np.sign(d.effect)) & ((d.brightness_adj_lo > 0) | (d.brightness_adj_hi < 0))).sum())}"
        f"/{len(d)}" for mk, d in rp.groupby("mark")}
print("reliable patient-level relations that survive the brightness adjustment:",
      OUT_JSON["reliable_P_surviving_brightness_adjustment"])
failed_reason = REL[REL.ci_excludes_0 & REL.large_enough & ~REL.reliable]
print("\nlarge, significant but not reliable:")
print(failed_reason[["relation", "effect", "basel", "zurich", "full_img", "eroded"]].round(3).to_string(index=False))
tick("phase A")

## Per-cell models (trained on the Jackson train patients; out-of-fold over all Jackson patients for power)

In [ ]:
FEATS = NUC + MARKERS + ["log_area"]

def feats(D):
    return rank_norm(np.column_stack([D["A"][c] for c in NUC + MARKERS] + [D["log_area"]]))

XJ, XD = feats(DJ), feats(DD)
SPECS = [("nuclear", "full", [FEATS.index(c) for c in NUC + ["log_area"]]),
         ("lgbm", "full", list(range(len(FEATS))))] + \
        [("lgbm", f"loa_{m}", [i for i, f in enumerate(FEATS) if f != m]) for m in MARKERS]
rng = np.random.default_rng(SEED + 1)
tr_rows = np.flatnonzero(DJ["split"] == "train")
te_rows = np.flatnonzero(DJ["split"] == "test")
tr_sub = np.sort(rng.choice(tr_rows, min(N_TRAIN, len(tr_rows)), replace=False))
folds = [(tr, te) for tr, te in GroupKFold(N_FOLD).split(np.zeros(DJ["n"]), groups=DJ["pcode"])]
fold_sub = [np.sort(rng.choice(tr, min(N_TRAIN, len(tr)), replace=False)) for tr, _ in folds]

def fit(cols, y, idx):
    m = lgb.LGBMRegressor(random_state=SEED, **LGB)
    m.fit(XJ[idx][:, cols], y[idx])
    return m

PJ, PJO, PD = {}, {}, {}      # (model, variant, mark) -> full-length predictions: Jackson test, Jackson out-of-fold, Danenberg
for mk, ch in EPI.items():
    y = DJ["A"][ch]
    for model, variant, cols in SPECS:
        m = fit(cols, y, tr_sub)
        a = np.full(DJ["n"], np.nan, np.float32)
        a[te_rows] = m.predict(XJ[te_rows][:, cols])
        PJ[(model, variant, mk)] = a
        PD[(model, variant, mk)] = m.predict(XD[:, cols]).astype(np.float32)
        o = np.full(DJ["n"], np.nan, np.float32)
        for (tr, te), sub in zip(folds, fold_sub):
            o[te] = fit(cols, y, sub).predict(XJ[te][:, cols])
        PJO[(model, variant, mk)] = o
    tick(f"per-cell models for {EPI_LABEL[mk]}")
pd.DataFrame({"image_name": DD["cells"].image_name.values, "object_number": DD["cells"].object_number.values,
              **{f"lgbm_{mk}": PD[("lgbm", "full", mk)] for mk in EPI}}).to_parquet(
    f"{OUT}/danenberg_epi_imputed_lgbm.parquet", index=False)

## Image-model predictions (`epi-immuvis-infer`), if available

In [ ]:
def align(D, T, col):
    idx = pd.MultiIndex.from_arrays([D["cells"].image_name.values, D["cells"].object_number.values])
    s = pd.Series(T[col].values.astype(np.float32), index=pd.MultiIndex.from_arrays([T.image_name.values, T.object_number.values]))
    return s.reindex(idx).values

def parse(col):
    parts = col.split("__")
    return (parts[0], parts[1], parts[2]) if len(parts) == 3 else (parts[0], "full", parts[1])

IMAGE_MODELS = []
pj_files, pd_files = g("epi_pred_jackson_test.parquet"), g("epi_pred_danenberg.parquet")
if SMOKE and not pj_files:          # synthetic stand-in, only to exercise the code path in a local test
    print("SMOKE: synthetic image-model predictions (truth + noise); not a result")
    rs = np.random.default_rng(0)
    TJ = DJ["cells"].iloc[te_rows][["image_name", "object_number"]].copy()
    TD = DD["cells"][["image_name", "object_number"]].copy()
    loa_img = TD.image_name.isin(TD.image_name.unique()[::2]).values
    for mk, ch in EPI.items():
        for v in ["full"] + [f"loa_{m}" for m in MARKERS]:
            TJ[f"immuvis__{v}__{mk}"] = DJ["A"][ch][te_rows] + rs.normal(0, 1, len(te_rows))
            TD[f"immuvis__{v}__{mk}"] = np.where(loa_img | (v == "full"),
                                                 PD[("lgbm", "full", mk)] + rs.normal(0, 0.5, len(TD)), np.nan)
        TJ[f"eva__{mk}"] = DJ["A"]["histone_h3"][te_rows] + rs.normal(0, 1, len(te_rows))
    pj_tabs, pd_tabs = [TJ], [TD]
else:
    pj_tabs = [read_parquet(f) for f in pj_files[:1]]
    pd_tabs = [read_parquet(f) for f in pd_files[:1]]
for T in pj_tabs:
    for col in [c for c in T.columns if c.startswith(("immuvis__", "eva__"))]:
        key = parse(col)
        a = align(DJ, T, col)
        assert np.isfinite(a[te_rows]).mean() > 0.95, (col, np.isfinite(a[te_rows]).mean())
        assert not np.isfinite(a[DJ["split"] != "test"]).any(), f"{col}: predictions outside the test patients"
        PJ[key] = a
for T in pd_tabs:
    for col in [c for c in T.columns if c.startswith(("immuvis__", "eva__"))]:
        PD[parse(col)] = align(DD, T, col)
IMAGE_MODELS = [m for m in ("immuvis", "eva") if any(k[0] == m for k in PJ)]
OUT_JSON["image_models"] = IMAGE_MODELS
print("image models found:", IMAGE_MODELS, "| Jackson-test columns:", sum(k[0] in IMAGE_MODELS for k in PJ),
      "| Danenberg columns:", sum(k[0] in IMAGE_MODELS for k in PD))
if IMAGE_MODELS:
    for f in g("run_info.json"):
        if "epi-immuvis-infer" in f.replace("\\", "/"):
            OUT_JSON["image_run_info"] = json.load(open(f))

## Accuracy in the Jackson test patients

In [ ]:
MODELS = ["nuclear", "lgbm"] + IMAGE_MODELS
full_keys = [(m, "full", mk) for m in MODELS for mk in EPI]
common = te_rows[np.isfinite(np.column_stack([PJ[k][te_rows] for k in full_keys])).all(1)]
VT = View(DJ, common, "Jackson test patients (cells with every prediction)")
VJO = View(DJ, np.arange(DJ["n"]), "Jackson, all patients, out-of-fold")
OUT_JSON["jackson_test"] = dict(cells=int(len(common)), patients=int(len(np.unique(DJ["pcode"][common]))),
                                images=int(len(np.unique(DJ["img"][common]))))
thr_phh3 = np.percentile(DJ["A"][EPI["phh3"]][tr_rows], 99)

def cell_auc(V, score, truth_hi):
    s, l = np.asarray(score)[V.rows], truth_hi[V.rows].astype(np.float64)
    point = float(_auc(s, l))
    pats = np.unique(V.pcode)
    by_p = pd.Series(np.arange(len(V.rows))).groupby(V.pcode).agg(list)
    rr = np.random.default_rng(SEED + 9)
    draws = []
    for _ in range(N_BOOT_AUC):
        ii = np.concatenate([by_p[p] for p in rr.choice(pats, len(pats))])
        draws.append(_auc(s[ii], l[ii]))
    return point, np.array(draws)

acc = []
for view_name, V, PRED, models in (("Jackson test", VT, PJ, MODELS), ("Jackson out-of-fold", VJO, PJO, ["nuclear", "lgbm"])):
    for mk, ch in EPI.items():
        e = DJ["A"][ch]
        for m in models:
            p = PRED[(m, "full", mk)]
            r, rb = V.r_within(e, p)
            pr, prb = V.cell_rel(V.ns(e), V.ns(p), "all")
            row = dict(evaluation=view_name, mark=mk, model=m, r_within=r, r_lo=ci(rb)[0], r_hi=ci(rb)[1],
                       partial_beyond_nuclear=pr, partial_lo=ci(prb)[0], partial_hi=ci(prb)[1])
            if m != "eva":
                sp, spb = spearman_boot(V.patient_mean(V.ns(p)), V.patient_mean(V.ns(e)))
                row.update(patient_spearman=sp, patient_lo=ci(spb)[0], patient_hi=ci(spb)[1])
            if mk == "phh3":
                au, aub = cell_auc(V, p, e > thr_phh3)
                row.update(auc_top1pct=au, auc_lo=ci(aub)[0], auc_hi=ci(aub)[1])
            acc.append(row)
ACC = pd.DataFrame(acc)
ACC.to_csv(f"{OUT}/accuracy.csv", index=False)
print(ACC.round(3).to_string(index=False))
OUT_JSON["accuracy"] = ACC.round(4).to_dict(orient="records")
tick("accuracy")

## Phase C: do the imputed marks keep the reliable relations?

In [ ]:
RELIABLE = [r for r in RELS if REL.set_index("id").loc[r["id"], "reliable"]]
REF_J = REL.set_index("id")
dan_keys = [(m, "full", mk) for m in MODELS for mk in EPI if (m, "full", mk) in PD]
d_ok = np.isfinite(np.column_stack([PD[k] for k in dan_keys])).all(1)
VD = View(DD, np.flatnonzero(d_ok), "Danenberg (cells with every prediction)")
VDL = None
loa_im = [k for k in PD if k[0] == "immuvis" and k[1].startswith("loa_")]
if loa_im:
    _r = np.flatnonzero(d_ok & np.isfinite(np.column_stack([PD[k] for k in loa_im])).all(1))
    print(f"Danenberg cells with every ImmuVis leave-anchor-out prediction: {len(_r):,}")
    if len(_r):
        VDL = View(DD, _r, "Danenberg, ImmuVis leave-anchor-out images")
OUT_JSON["danenberg_cells"] = dict(all=int(d_ok.sum()), loa_subset=int(len(VDL.rows)) if VDL is not None else 0)

def variants_for(rel, model):
    if model == "eva" and rel["family"] not in "CK":
        return []
    v = ["full"]
    if rel["anchor"] is not None and model in ("lgbm", "immuvis"):
        v.append(f"loa_{rel['anchor']}")
    return v

def judge(pt, bt, ref):
    lo, hi = ci(bt)
    ratio = pt / ref if ref else np.nan
    return dict(effect=pt, lo=lo, hi=hi, ratio=ratio,
                reproduced=bool(np.sign(pt) == np.sign(ref) and (lo > 0 or hi < 0) and RATIO[0] <= ratio <= RATIO[1]))

fid = []
for rel in RELIABLE:
    ch = EPI[rel["mark"]]
    ref_t, ref_tb = rel_value(VT, rel, DJ["A"][ch])
    ref_o, ref_ob = rel_value(VJO, rel, DJ["A"][ch])
    ref_j = float(REF_J.loc[rel["id"], "effect"])
    for m in MODELS:
        for v in variants_for(rel, m):
            key = (m, v, rel["mark"])
            base = dict(id=rel["id"], family=rel["family"], mark=rel["mark"], relation=describe(rel), model=m, variant=v,
                        primary=(v != "full") == (rel["anchor"] is not None and m in ("lgbm", "immuvis")))
            if key in PJ:
                pt, bt = rel_value(VT, rel, PJ[key])
                fid.append(dict(base, evaluation="Jackson test", reference=ref_t,
                                diff_lo=ci(bt - ref_tb)[0], diff_hi=ci(bt - ref_tb)[1], **judge(pt, bt, ref_t)))
            if key in PJO:
                pt, bt = rel_value(VJO, rel, PJO[key])
                fid.append(dict(base, evaluation="Jackson out-of-fold", reference=ref_o,
                                diff_lo=ci(bt - ref_ob)[0], diff_hi=ci(bt - ref_ob)[1], **judge(pt, bt, ref_o)))
            if key in PD:
                Vd = VDL if (m == "immuvis" and v != "full") else VD
                if Vd is not None:
                    pt, bt = rel_value(Vd, rel, PD[key])
                    fid.append(dict(base, evaluation="Danenberg", reference=ref_j, diff_lo=np.nan, diff_hi=np.nan,
                                    **judge(pt, bt, ref_j)))
FID = pd.DataFrame(fid)
FID.to_csv(f"{OUT}/relation_fidelity.csv", index=False)
tick(f"fidelity: {len(RELIABLE)} reliable relations, {len(FID)} rows")

## Verdict

In [ ]:
ver = []
for mk in EPI:
    for m in MODELS:
        a = ACC[(ACC.evaluation == "Jackson test") & (ACC.model == m) & (ACC.mark == mk)].iloc[0]
        A = bool(a.partial_beyond_nuclear >= ACC_MIN and a.partial_lo > 0)
        row = dict(mark=mk, model=m, partial_beyond_nuclear=a.partial_beyond_nuclear, A_accuracy=A)
        for tag, ev in (("B_jackson_test", "Jackson test"), ("C_danenberg", "Danenberg"), ("jackson_oof", "Jackson out-of-fold")):
            f = FID[(FID.model == m) & (FID.mark == mk) & (FID.evaluation == ev) & FID.primary]
            row[f"{tag}_reproduced"] = int(f.reproduced.sum())
            row[f"{tag}_of"] = int(len(f))
            row[tag] = bool(len(f) > 0 and f.reproduced.mean() >= REPRO_FRAC)
            ff = FID[(FID.model == m) & (FID.mark == mk) & (FID.evaluation == ev) & (FID.variant == "full")]
            row[f"{tag}_full_inputs_reproduced"] = f"{int(ff.reproduced.sum())}/{len(ff)}"
        if m == "eva":
            row["verdict"] = "passes A and B (not run on Danenberg)" if A and row["B_jackson_test"] else "no"
        else:
            row["verdict"] = ("infers" if A and row["B_jackson_test"] and row["C_danenberg"] else
                              "only within Jackson" if A and row["B_jackson_test"] else "no")
        ver.append(row)
VER = pd.DataFrame(ver)
VER.to_csv(f"{OUT}/verdict.csv", index=False)
print(VER.to_string(index=False))
OUT_JSON["verdict"] = VER.round(4).to_dict(orient="records")

## Amendment 1: Danenberg fidelity on anchors that transfer between cohorts (post hoc)

In [ ]:
TRANSFER_J_TO_D = {      # train-test-grid: R4 (Danenberg test, cell-mean truth, Table, mean of 3 replicates), Jackson-trained / Danenberg-trained
    "sma": -0.214, "cd68": 0.427, "cd20": 0.506, "cytokeratin_5": 0.733, "estrogen_receptor_alpha": 0.803, "ki_67": 0.821,
    "cd45": 0.858, "c_erb_b_2_her2": 0.891, "cd3": 0.897, "pan_cytokeratin": 0.967, "cytokeratin_8_18": 1.008}
TRANSFER_MIN = 0.60
assert set(TRANSFER_J_TO_D) == set(MARKERS)

def anchor_of(rel_id):
    p = rel_id.split("|")
    return p[2] if p[0] in ("C", "P") else None

FID["anchor_transfers"] = [anchor_of(i) is None or TRANSFER_J_TO_D[anchor_of(i)] >= TRANSFER_MIN for i in FID.id]
FID.to_csv(f"{OUT}/relation_fidelity.csv", index=False)
am = []
for mk in EPI:
    for m in MODELS:
        f = FID[(FID.model == m) & (FID.mark == mk) & (FID.evaluation == "Danenberg") & FID.primary & FID.anchor_transfers]
        am.append(dict(mark=mk, model=m, C2_danenberg_transferable_reproduced=int(f.reproduced.sum()),
                       C2_danenberg_transferable_of=int(len(f)),
                       C2_danenberg_transferable=bool(len(f) > 0 and f.reproduced.mean() >= REPRO_FRAC)))
VER = VER.merge(pd.DataFrame(am), on=["mark", "model"])
VER["verdict_amended"] = [v.verdict if v.model == "eva" else
                          "infers" if v.A_accuracy and v.B_jackson_test and v.C2_danenberg_transferable else
                          "only within Jackson" if v.A_accuracy and v.B_jackson_test else "no" for v in VER.itertuples()]
VER.to_csv(f"{OUT}/verdict.csv", index=False)
OUT_JSON["verdict"] = VER.round(4).to_dict(orient="records")
OUT_JSON["amendment_1"] = dict(transfer_ratio=TRANSFER_J_TO_D, min_ratio=TRANSFER_MIN,
                               excluded_anchors=sorted(m for m, r in TRANSFER_J_TO_D.items() if r < TRANSFER_MIN))
print(VER[["mark", "model", "A_accuracy", "B_jackson_test", "C_danenberg_reproduced", "C_danenberg_of",
           "C2_danenberg_transferable_reproduced", "C2_danenberg_transferable_of", "verdict", "verdict_amended"]].to_string(index=False))

## Keren 2018: H3K27me3 against the measured mark (pre-registered)

In [ ]:
KEREN_OK = False
try:
    assert "keren" in P, "keren-cells-prep output not attached"
    K_MAP = {"dna1": "ds_dna", "ki_67": "ki67", "cd3": "cd3", "cd20": "cd20", "cd45": "cd45", "cd68": "cd68",
             "sma": "sma", "pan_cytokeratin": "pan_keratin", EPI["h3k27me3"]: "h3k27me3"}     # Jackson name -> Keren name
    K_MARKERS = ["ki_67", "cd3", "cd20", "cd45", "cd68", "sma", "pan_cytokeratin"]
    K_NUC = ["dna1", "log_area"]          # Keren: dsDNA (stands in for DNA1) and log area; no H3 or DNA2
    K_FEATS = ["dna1"] + K_MARKERS + ["log_area"]
    MK, CH_K = "h3k27me3", EPI["h3k27me3"]
    kc = read_parquet(P["keren"]["cells"])
    n_per = kc.groupby("image_name").size()
    kc = kc[kc.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    kc = kc.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    tr = lambda v: np.arcsinh(np.asarray(v, np.float64) / COFACTOR).astype(np.float32)
    DK = dict(key="keren", cells=kc[["image_name", "object_number"]].assign(patient=kc.patient_id.astype(str),
                                                                           sub="keren", broad=kc.broad_class.astype(str)),
              A={j: tr(kc[k]) for j, k in K_MAP.items()}, E={j: tr(kc[f"eroded__{k}"]) for j, k in K_MAP.items()},
              log_area=np.log(np.maximum(kc.area.values.astype(np.float64), 1)).astype(np.float32),
              img=pd.factorize(kc.image_name, sort=True)[0].astype(np.int64), broad=kc.broad_class.values.astype(str),
              sub=np.full(len(kc), "keren"), clin={})
    DK["pats"], DK["pcode"] = np.unique(DK["cells"].patient.values, return_inverse=True)
    DK["n"] = len(kc)
    print(f"keren: {DK['n']:,} cells, {len(DK['pats'])} patients, classes {pd.Series(DK['broad']).value_counts().to_dict()}")

    # --- relations reliable in Jackson, with the Keren nuclear covariates
    K_RELS = [r for r in RELS if r["mark"] == MK and r["family"] in "CKP" and (r["anchor"] is None or r["anchor"] in K_MARKERS)]
    VJk, VBk, VZk = (View(DJ, rows, nm, nuc=K_NUC) for rows, nm in (
        (np.arange(DJ["n"]), "Jackson, all (Keren covariates)"), (np.flatnonzero(DJ["sub"] == "basel"), "Basel"),
        (np.flatnonzero(DJ["sub"] == "zurich"), "Zurich")))
    tnbc_p = DJ["clin"]["tnbc"] == 1
    VJt = View(DJ, np.flatnonzero(tnbc_p[DJ["pcode"]]), "Jackson, triple-negative patients", nuc=K_NUC)
    VJFk = VJEk = None
    if IMG:
        VJFk = View(DJ, VJF.rows, "Jackson image values, full masks (Keren covariates)", src=IMG["full"], nuc=K_NUC)
        VJEk = View(DJ, VJE.rows, "Jackson image values, eroded masks (Keren covariates)", src=IMG["eroded"], nuc=K_NUC)
    VK = View(DK, np.arange(DK["n"]), "Keren, all cells", nuc=K_NUC)
    ok_k = np.isfinite(np.column_stack([DK["E"][c] for c in DK["E"]])).all(1)
    VKF = View(DK, np.flatnonzero(ok_k), "Keren, full masks", nuc=K_NUC)
    VKE = View(DK, np.flatnonzero(ok_k), "Keren, eroded masks", src={**DK["E"], "log_area": DK["log_area"]}, nuc=K_NUC)
    rows = []
    for rel in K_RELS:
        pt, bt = rel_value(VJk, rel, DJ["A"][CH_K])
        row = dict(id=rel["id"], family=rel["family"], anchor=rel["anchor"], relation=describe(rel), jackson=pt,
                   jackson_lo=ci(bt)[0], jackson_hi=ci(bt)[1], basel=rel_value(VBk, rel, DJ["A"][CH_K])[0],
                   zurich=rel_value(VZk, rel, DJ["A"][CH_K])[0], jackson_tnbc=rel_value(VJt, rel, DJ["A"][CH_K])[0],
                   full_img=np.nan, eroded=np.nan)
        if rel["family"] in "CK" and VJEk is not None:
            row["full_img"] = rel_value(VJFk, rel, IMG["full"][CH_K])[0]
            row["eroded"] = rel_value(VJEk, rel, IMG["eroded"][CH_K])[0]
        km, kb = rel_value(VK, rel, DK["A"][CH_K])
        row.update(keren_measured=km, keren_lo=ci(kb)[0], keren_hi=ci(kb)[1],
                   replicated_measured=judge(km, kb, pt)["reproduced"])
        if rel["family"] in "CK":
            row["keren_full_mask"] = rel_value(VKF, rel, DK["A"][CH_K])[0]
            row["keren_eroded"] = rel_value(VKE, rel, DK["E"][CH_K])[0]
        rows.append(row)
    KREL = pd.DataFrame(rows)
    s = np.sign(KREL.jackson)
    KREL["reliable"] = (((KREL.jackson_lo > 0) | (KREL.jackson_hi < 0)) & (KREL.jackson.abs() >= KREL.family.map(THR)) &
                        (np.sign(KREL.basel) == s) & (np.sign(KREL.zurich) == s) &
                        np.where(KREL.family.isin(["C", "K"]), (np.sign(KREL.eroded) == s) & (np.sign(KREL.full_img) == s) &
                                 (KREL.eroded.abs() >= 0.5 * KREL.full_img.abs()), True))
    KREL["keren_erosion_ok"] = np.where(KREL.family.isin(["C", "K"]),
                                        (np.sign(KREL.keren_eroded) == np.sign(KREL.keren_full_mask)) &
                                        (KREL.keren_eroded.abs() >= 0.5 * KREL.keren_full_mask.abs()), np.nan)

    # --- models on the Keren inputs (Jackson train patients), applied to the Jackson test patients and to Keren
    def kfeats(D):
        return rank_norm(np.column_stack([D["A"][c] for c in K_FEATS[:-1]] + [D["log_area"]]))
    XJk, XK = kfeats(DJ), kfeats(DK)
    KSPECS = [("nuclear", "full", [0, len(K_FEATS) - 1]), ("lgbm", "full", list(range(len(K_FEATS))))] + \
             [("lgbm", f"loa_{m}", [i for i, f in enumerate(K_FEATS) if f != m]) for m in K_MARKERS]
    PKJ, PKK = {}, {}
    for model, variant, cols in KSPECS:
        m = lgb.LGBMRegressor(random_state=SEED, **LGB).fit(XJk[tr_sub][:, cols], DJ["A"][CH_K][tr_sub])
        a = np.full(DJ["n"], np.nan, np.float32)
        a[te_rows] = m.predict(XJk[te_rows][:, cols])
        PKJ[(model, variant)], PKK[(model, variant)] = a, m.predict(XK[:, cols]).astype(np.float32)
    tick("Keren-input models")

    # --- does each anchor transfer from Jackson to Keren?
    kfolds = list(GroupKFold(N_FOLD).split(np.zeros(DK["n"]), groups=DK["pcode"]))
    tx = []
    for M in K_MARKERS:
        j = K_FEATS.index(M)
        cols = [i for i in range(len(K_FEATS)) if i != j]
        yk = XK[:, j]
        pj = lgb.LGBMRegressor(random_state=SEED, **LGB).fit(XJk[tr_sub][:, cols], XJk[tr_sub][:, j]).predict(XK[:, cols])
        pk = np.full(DK["n"], np.nan)
        for trk, tek in kfolds:
            sub = trk if len(trk) <= N_TRAIN else np.sort(np.random.default_rng(SEED).choice(trk, N_TRAIN, replace=False))
            pk[tek] = lgb.LGBMRegressor(random_state=SEED, **LGB).fit(XK[sub][:, cols], yk[sub]).predict(XK[tek][:, cols])
        rj, rk_ = VK.r_within(yk, pj)[0], VK.r_within(yk, pk)[0]
        tx.append(dict(anchor=M, r_jackson_trained=rj, r_keren_trained=rk_, ratio=rj / rk_ if rk_ > 0 else np.nan))
    KTX = pd.DataFrame(tx)
    KTX["transfers"] = KTX.ratio >= TRANSFER_MIN
    KTX.to_csv(f"{OUT}/keren_anchor_transfer.csv", index=False)
    print(KTX.round(3).to_string(index=False))
    K_TRANSFERS = set(KTX.anchor[KTX.transfers])
    KREL["anchor_transfers"] = [a is None or (isinstance(a, str) and a in K_TRANSFERS) for a in KREL.anchor]
    KREL.to_csv(f"{OUT}/keren_relations.csv", index=False)
    print(KREL.round(3).to_string(index=False))

    # --- accuracy: Keren, and the same models in the Jackson test patients
    VTk = View(DJ, te_rows, "Jackson test (Keren covariates)", nuc=K_NUC)
    kacc = []
    for ev, V, e, PR in (("Keren", VK, DK["A"][CH_K], PKK), ("Jackson test", VTk, DJ["A"][CH_K], PKJ)):
        for m in ("nuclear", "lgbm"):
            p = PR[(m, "full")]
            r, rb = V.r_within(e, p)
            pr, prb = V.cell_rel(V.ns(e), V.ns(p), "all")
            sp, spb = spearman_boot(V.patient_mean(V.ns(p)), V.patient_mean(V.ns(e)))
            kacc.append(dict(evaluation=ev, model=m, r_within=r, r_lo=ci(rb)[0], r_hi=ci(rb)[1], partial_beyond_nuclear=pr,
                             partial_lo=ci(prb)[0], partial_hi=ci(prb)[1], patient_spearman=sp, patient_lo=ci(spb)[0],
                             patient_hi=ci(spb)[1]))
    KACC = pd.DataFrame(kacc)
    KACC.to_csv(f"{OUT}/keren_accuracy.csv", index=False)
    print(KACC.round(3).to_string(index=False))

    # --- fidelity of the imputed mark on Keren (truth-referenced and Jackson-referenced) and in the Jackson test patients
    kf = []
    for rel in [r for r in K_RELS if bool(KREL.set_index("id").loc[r["id"], "reliable"])]:
        kr = KREL.set_index("id").loc[rel["id"]]
        ref_t, _ = rel_value(VTk, rel, DJ["A"][CH_K])
        for m in ("nuclear", "lgbm"):
            for v in ["full"] + ([f"loa_{rel['anchor']}"] if rel["anchor"] is not None and m == "lgbm" else []):
                base = dict(id=rel["id"], family=rel["family"], relation=describe(rel), model=m, variant=v,
                            primary=(v != "full") == (rel["anchor"] is not None and m == "lgbm"),
                            anchor_transfers=bool(kr.anchor_transfers))
                pt, bt = rel_value(VK, rel, PKK[(m, v)])
                kf.append(dict(base, evaluation="Keren vs measured", reference=kr.keren_measured, **judge(pt, bt, kr.keren_measured)))
                kf.append(dict(base, evaluation="Keren vs Jackson", reference=kr.jackson, **judge(pt, bt, kr.jackson)))
                pt, bt = rel_value(VTk, rel, PKJ[(m, v)])
                kf.append(dict(base, evaluation="Jackson test", reference=ref_t, **judge(pt, bt, ref_t)))
    KFID = pd.DataFrame(kf)
    KFID.to_csv(f"{OUT}/keren_fidelity.csv", index=False)

    kv = []
    for m in ("nuclear", "lgbm"):
        a = KACC[(KACC.evaluation == "Keren") & (KACC.model == m)].iloc[0]
        A = bool(a.partial_beyond_nuclear >= ACC_MIN and a.partial_lo > 0)
        f = KFID[(KFID.model == m) & KFID.primary]
        row = dict(model=m, partial_beyond_nuclear_keren=a.partial_beyond_nuclear, A_keren=A)
        for tag, ev, sel in (("D_keren", "Keren vs measured", True), ("D_keren_all_anchors", "Keren vs measured", False),
                             ("keren_vs_jackson", "Keren vs Jackson", True), ("jackson_test", "Jackson test", True)):
            ff = f[(f.evaluation == ev) & (f.anchor_transfers | (not sel))]
            row[f"{tag}_reproduced"], row[f"{tag}_of"] = int(ff.reproduced.sum()), int(len(ff))
            row[tag] = bool(len(ff) > 0 and ff.reproduced.mean() >= REPRO_FRAC)
        t_ = f[f.evaluation == "Keren vs measured"].set_index("id").reproduced
        j_ = f[f.evaluation == "Keren vs Jackson"].set_index("id").reproduced.reindex(t_.index)
        row["proxy_agreement"] = f"{int((t_ == j_).sum())}/{len(t_)}"
        row["proxy_false_pass"] = int((j_ & ~t_).sum())
        row["proxy_false_fail"] = int((~j_ & t_).sum())
        row["verdict_keren"] = ("infers in Keren" if A and row["D_keren"] else
                                "accurate, relations not kept" if A else "no")
        kv.append(row)
    KVER = pd.DataFrame(kv)
    KVER.to_csv(f"{OUT}/keren_verdict.csv", index=False)
    print(KVER.to_string(index=False))
    rel_ok = KREL[KREL.reliable]
    OUT_JSON["keren"] = dict(
        cells=DK["n"], patients=len(DK["pats"]), candidate_relations=len(KREL), reliable_in_jackson=int(len(rel_ok)),
        reliable_with_transferable_anchor=int(rel_ok.anchor_transfers.sum()),
        replicated_with_measured_mark=f"{int(rel_ok.replicated_measured.sum())}/{len(rel_ok)}",
        replicated_with_measured_mark_transferable=f"{int(rel_ok[rel_ok.anchor_transfers].replicated_measured.sum())}"
                                                   f"/{int(rel_ok.anchor_transfers.sum())}",
        anchors_transferring=sorted(K_TRANSFERS), accuracy=KACC.round(4).to_dict(orient="records"),
        verdict=KVER.round(4).to_dict(orient="records"))
    print(json.dumps({k: v for k, v in OUT_JSON["keren"].items() if k not in ("accuracy", "verdict")}, indent=1))
    KEREN_OK = True
    tick("Keren")
except Exception:
    print("!!! Keren block failed:\n" + traceback.format_exc(), flush=True)
    FAILED.append("keren")

## Figures

In [ ]:
COL = {"nuclear": "#9e9e9e", "lgbm": "#1f4e9c", "immuvis": "#3d85d6", "eva": "#9cc3ee"}
FILLED = {"nuclear": False, "lgbm": False, "immuvis": True, "eva": True}
try:
    fig, axes = plt.subplots(2, 2, figsize=(9, 6.5))
    T = ACC[ACC.evaluation == "Jackson test"]
    for i, mk in enumerate(EPI):
        for j, (metric, lo, hi, ylab) in enumerate((
                ("r_within", "r_lo", "r_hi", "Cell correlation within each image"),
                ("partial_beyond_nuclear", "partial_lo", "partial_hi", "Partial correlation,\nbeyond nuclear content"))):
            ax = axes[i, j]
            d = T[T.mark == mk].set_index("model").reindex(MODELS)
            x = np.arange(len(MODELS))
            for k, m in enumerate(MODELS):
                ax.bar(x[k], d.loc[m, metric], width=0.7, color=COL[m] if FILLED[m] else "white",
                       edgecolor=COL[m], linewidth=1.5)
                ax.errorbar(x[k], d.loc[m, metric], yerr=[[d.loc[m, metric] - d.loc[m, lo]], [d.loc[m, hi] - d.loc[m, metric]]],
                            color="black", capsize=3, lw=1)
            ax.set_xticks(x)
            ax.set_xticklabels([MODEL_NAME[m].replace(", ", ",\n") for m in MODELS], fontsize=8)
            ax.axhline(0, color="black", lw=0.8)
            ax.set_title(EPI_LABEL[mk], fontsize=10)
            ax.set_ylabel(ylab, fontsize=9)
            ax.set_xlabel("Jackson test patients", fontsize=8)
            ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(f"{OUT}/fig_epi_accuracy.png", dpi=200)
    plt.close(fig)

    SHAPE = {"C": "o", "K": "^", "P": "s", "L": "D"}
    evs = ["Jackson test", "Danenberg"]
    fig, axes = plt.subplots(2, 2, figsize=(9, 8.5))
    for i, mk in enumerate(EPI):
        for j, ev in enumerate(evs):
            ax = axes[i, j]
            f = FID[(FID.mark == mk) & (FID.evaluation == ev) & FID.primary]
            lim = max(0.2, float(np.nanmax(np.abs(np.r_[f.reference.values, f.effect.values, 0.2])))) * 1.1 if len(f) else 1
            xs = np.linspace(-lim, lim, 50)
            ax.fill_between(xs, 0.5 * xs, 2 * xs, color="#eeeeee", zorder=0)
            ax.plot([-lim, lim], [-lim, lim], color="black", lw=0.8)
            for m in MODELS:
                for fam, mkr in SHAPE.items():
                    d = f[(f.model == m) & (f.family == fam)]
                    if len(d):
                        ax.scatter(d.reference, d.effect, marker=mkr, s=28, linewidths=1.2, edgecolors=COL[m],
                                   facecolors=COL[m] if FILLED[m] else "none", zorder=3)
            ax.axhline(0, color="black", lw=0.5)
            ax.axvline(0, color="black", lw=0.5)
            ax.set_xlim(-lim, lim)
            ax.set_ylim(-lim, lim)
            ax.set_xlabel(f"Effect with measured {EPI_LABEL[mk]} " +
                          ("(same Jackson test cells)" if ev == "Jackson test" else "(all Jackson patients)"), fontsize=8)
            ax.set_ylabel(f"Effect with imputed {EPI_LABEL[mk]} ({ev})", fontsize=8)
            ax.spines[["top", "right"]].set_visible(False)
    handles = [plt.Line2D([], [], marker="o", ls="", markeredgecolor=COL[m], markerfacecolor=COL[m] if FILLED[m] else "none",
                          label=MODEL_NAME[m]) for m in MODELS]
    handles += [plt.Line2D([], [], marker=s, ls="", color="black", markerfacecolor="none", label=lab) for s, lab in
                (("o", "cells"), ("^", "cell classes"), ("s", "patients"), ("D", "clinical"))]
    fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False, fontsize=8)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    fig.savefig(f"{OUT}/fig_epi_relations.png", dpi=200)
    plt.close(fig)
except Exception:
    print("!!! figures failed:\n" + traceback.format_exc(), flush=True)
    FAILED.append("figures")



In [ ]:
if KEREN_OK:
    try:
        SHAPE = {"C": "o", "K": "^", "P": "s"}
        fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
        d = KREL[KREL.reliable]
        lim = max(0.2, float(np.nanmax(np.abs(np.r_[d.jackson, d.keren_measured])))) * 1.1
        f = KFID[(KFID.model == "lgbm") & KFID.primary & (KFID.evaluation == "Keren vs measured")]
        lim2 = max(0.2, float(np.nanmax(np.abs(np.r_[f.reference, f.effect])))) * 1.1 if len(f) else 1
        for ax, L in ((axes[0], lim), (axes[1], lim2)):
            xs = np.linspace(-L, L, 50)
            ax.fill_between(xs, 0.5 * xs, 2 * xs, color="#eeeeee", zorder=0)
            ax.plot([-L, L], [-L, L], color="black", lw=0.8)
            ax.axhline(0, color="black", lw=0.5)
            ax.axvline(0, color="black", lw=0.5)
            ax.set_xlim(-L, L)
            ax.set_ylim(-L, L)
            ax.spines[["top", "right"]].set_visible(False)
        for fam, mkr in SHAPE.items():
            for tr_, fc in ((True, "#1f4e9c"), (False, "none")):
                x = d[(d.family == fam) & (d.anchor_transfers == tr_)]
                axes[0].scatter(x.jackson, x.keren_measured, marker=mkr, s=30, edgecolors="#1f4e9c", facecolors=fc, zorder=3)
                y = f[(f.family == fam) & (f.anchor_transfers == tr_)]
                axes[1].scatter(y.reference, y.effect, marker=mkr, s=30, edgecolors="#1f4e9c", facecolors=fc, zorder=3)
        axes[0].set_xlabel("H3K27me3 relation, Jackson (measured)", fontsize=9)
        axes[0].set_ylabel("H3K27me3 relation, Keren (measured)", fontsize=9)
        axes[1].set_xlabel("H3K27me3 relation, Keren (measured)", fontsize=9)
        axes[1].set_ylabel("H3K27me3 relation, Keren (imputed, LightGBM)", fontsize=9)
        handles = [plt.Line2D([], [], marker=s, ls="", color="black", markerfacecolor="none", label=lab) for s, lab in
                   (("o", "cells"), ("^", "cell classes"), ("s", "patients"))]
        handles += [plt.Line2D([], [], marker="o", ls="", markeredgecolor="#1f4e9c", markerfacecolor=fc, label=lab)
                    for fc, lab in (("#1f4e9c", "anchor transfers"), ("none", "anchor does not transfer"))]
        fig.legend(handles=handles, loc="upper center", ncol=5, frameon=False, fontsize=8)
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        fig.savefig(f"{OUT}/fig_epi_keren.png", dpi=200)
        plt.close(fig)
    except Exception:
        print("!!! Keren figure failed:\n" + traceback.format_exc(), flush=True)
        FAILED.append("keren figure")

In [ ]:
OUT_JSON.update(failed_blocks=FAILED, minutes=(time.time() - T0) / 60,
                thresholds=dict(THR=THR, ratio=RATIO, acc_min=ACC_MIN, repro_frac=REPRO_FRAC, phh3_top1pct=float(thr_phh3)))
json.dump(OUT_JSON, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
print("failed blocks:", FAILED if FAILED else "none")